# GameTheory 16e - Tour de SocialChoiceLean (DominikPeters)

**Navigation** : [16b-Lean-SocialChoice](GameTheory-16b-Lean-SocialChoice.ipynb) | [16c-Python](GameTheory-16c-SocialChoice-Python.ipynb) | [16d-SAT](GameTheory-16d-SocialChoice-SAT.ipynb) | **16e-PetersTour**

---

## Introduction

Ce notebook presente les resultats principaux formalises dans le depot [DominikPeters/SocialChoiceLean](https://github.com/DominikPeters/SocialChoiceLean), une bibliotheque Lean 4 couvrant :

- **Theoremes d'impossibilite** : Gibbard-Satterthwaite, Duggan-Schwartz, 4 impossibilites de Condorcet, Holliday
- **15+ regles de vote** : Split Cycle, Schulze, Black, Copeland, Minimax, Borda, Plurality, IRV, Top Cycle, Uncovered Set, etc.
- **Verification d'axiomes** : chaque regle est verifiee contre 15+ axiomes (Condorcet, Monotonicite, Pareto, Neutrality, etc.)

### Framework vs notre port

| Aspect | Notre port (asouther4/chasenorman) | DominikPeters |
|--------|-------------------------------------|---------------|
| Langage | Lean 4 (v4.28.0-rc1) | Lean 4 (v4.27.0-rc1) |
| Preferences | `PrefOrder` (reflexive + total + transitive) | `LinearOrder` (Mathlib class) |
| Profils | `Profile := ι → PrefOrder σ` | `Profile V A` structure |
| Regles | `SCC ι σ` (correspondence) | `VotingRule` (polymorphe) |
| Scope | Arrow + Sen + Voting | 15+ regles + 7 impossibilites |
| Preuves | Manuelles (tactiques Lean) | GPT-5.2 + verification Lean |

Le projet Lake reference est dans `social_choice_lean_peters/`.

---

## 1. Cadre Formel (DominikPeters)

### 1.1 Profils et Regles de Vote

DominikPeters utilise un cadre base sur `LinearOrder` de Mathlib, ou chaque electeur a un **ordre total strict** (pas de preferences faibles) :

```lean
-- From DominikPeters/SocialChoiceLean/SocialChoice/Profile.lean

-- Un profil associe a chaque electeur un ordre lineaire strict
structure Profile (V A : Type) [Fintype V] [Fintype A] where
  pref : V → LinearOrder A

-- Une regle de vote est polymorphe sur les types d'electeurs et de candidats
abbrev VotingRule :=
  ∀ {V A : Type} [Fintype V] [Fintype A], Profile V A → Finset A

-- Preference individuelle : P.pref v est un LinearOrder, donc .lt donne la preference
def Prefers {V A : Type} [Fintype V] [Fintype A]
    (P : Profile V A) (v : V) (a b : A) : Prop :=
  (P.pref v).lt a b

-- Rang superieur : un candidat c est premier pour l'electeur v
def TopRank {V A : Type} [Fintype V] [Fintype A]
    (P : Profile V A) (v : V) (c : A) : Prop :=
  ∀ d : A, d ≠ c → Prefers P v c d
```

### 1.2 Fonction de Marge

La **marge** d'un candidat sur un autre mesure la difference de votes :

```lean
-- From DominikPeters/SocialChoiceLean/SocialChoice/Margin.lean

-- Marge de a sur b : nombre d'electeurs preferant a a b moins reciproquement
noncomputable def margin {V A : Type} [Fintype V] [Fintype A]
    (P : Profile V A) (a b : A) : Int :=
  (Finset.univ.filter (fun v => Prefers P v a b)).card -
    (Finset.univ.filter (fun v => Prefers P v b a)).card

def margin_pos {V A : Type} [Fintype V] [Fintype A]
    (P : Profile V A) (a b : A) : Prop :=
  0 < margin P a b
```

---

## 2. Axiomes de Vote

DominikPeters definit et utilise 15+ axiomes, marques avec `@[scAxiom]` pour le meta-framework :

```lean
-- From DominikPeters/SocialChoiceLean/SocialChoice/Axioms/

-- Pareto : si tous preferent c a d, d ne peut pas gagner
@[scAxiom]
def ParetoEfficiency (f : VotingRule) : Prop :=
  ∀ {V A} [Fintype V] [Fintype A] [Nonempty V] (P : Profile V A) (c d : A),
    (∀ v, Prefers P v c d) → d ∉ f P

-- Vainqueur de Condorcet : bat tous les autres par majorite stricte
def CondorcetWinner {V A : Type} [Fintype V] [Fintype A]
    (P : Profile V A) (c : A) : Prop :=
  ∀ d : A, d ≠ c → StrictMajority (votersPreferring P c d)

-- Consistance de Condorcet : le vainqueur de Condorcet est l'unique gagnant
@[scAxiom]
def CondorcetConsistency (f : VotingRule) : Prop :=
  ∀ {V A} [Fintype V] [Fintype A] (P : Profile V A) (c : A),
    CondorcetWinner P c → f P = {c}

-- Dictature : un electeur determine toujours le resultat
@[scAxiom]
def Dictatorial (f : VotingRule) (_hf : Resolute f) : Prop :=
  ∀ {V A} [Fintype V] [Fintype A] [Nonempty A],
    ∃ d : V, ∀ P : Profile V A, f P = {topChoice P d}
```

---

## 3. Theoreme de Gibbard-Satterthwaite

Le resultat central : toute regle de vote resolutive, unanime et non-manipulable
(pour >= 3 candidats) doit etre une dictature.

```lean
-- From DominikPeters/SocialChoiceLean/SocialChoice/Impossibilities/GibbardSatterthwaite/Main.lean

/-- **Gibbard-Satterthwaite (1973/1975)** : Toute regle resolutive, unanime et
    non-manipulable (>= 3 candidats) est dictatrice.

    Preuve par induction forte sur le nombre d'electeurs :
    - Base (1 electeur) : trivialement dictatorial
    - Induction : clonage d'un electeur, application de l'hypothese de recurrence
      sur l'electorat reduit, puis analyse de cas sur l'identite du dictateur -/
theorem gibbard_satterthwaite
    {V A : Type} [Fintype V] [Nonempty V] [Fintype A] [Nonempty A]
    (hcardA : 3 ≤ Fintype.card A)
    (f : VotingRule)
    (hf_res : Resolute f)
    (hf_unan : Unanimity f)
    (hf_sp : ResoluteStrategyproofness f hf_res) :
    ∃ d : V, ∀ P : Profile V A, f P = {topChoice P d}
```

### Signification

Ce theoreme montre que **toute election est manipulable** (sauf dictature) :
- Si la regle est resolutive (1 seul gagnant)
- Et unanime (si tout le monde met c en tete, c gagne)
- Et non-manipulable (aucun electeur ne peut ameliorer le resultat en mentant)
- Alors il existe un dictateur dont le choix prevaut toujours

La preuve formelle fait 5 fichiers Lean (BaseCase + Common + InductionStepCase1 + InductionStepCase2 + Main).

---

## 4. Impossibilites de Condorcet

DominikPeters formalise 4 theoremes d'impossibilite lies a Condorcet :

### 4.1 Condorcet + Participation => Impossible (Moulin 1988)

**Enonce** : Aucune regle de vote ne peut simultanement etre coherente avec Condorcet
et satisfaire la participation (ajouter des electeurs qui classent c en tete
ne doit pas faire perdre c).

Formalise dans `SocialChoice.Impossibilities.CondorcetParticipation`.

### 4.2 Condorcet + Renforcement => Impossible (Young 1975)

**Enonce** : Aucune regle ne peut etre coherente avec Condorcet et satisfaire le
renforcement (si deux electorats disjoints donnent le meme gagnant, l'union aussi).

Formalise dans `SocialChoice.Impossibilities.CondorcetReinforcement`.

### 4.3 Condorcet + Manipulabilite => Impossible

**Enonce** : La coherence de Condorcet est incompatible avec la non-manipulabilite.

Formalise dans `SocialChoice.Impossibilities.CondorcetStrategyproofness`.

### 4.4 Anonymat + Neutrite + Resolutude => Impossible (nombre pair d'electeurs)

**Enonce** : Pour un nombre pair d'electeurs, anonymat + neutralite + regle resolutive
sont incompatibles.

Formalise dans `SocialChoice.Impossibilities.AnonymousNeutralResolute`.

---

## 5. Split Cycle (Holliday & Pacuit 2023)

La regle Split Cycle est un cas particulier : c'est la regle la plus fine qui
soit coherente avec Condorcet et acyclique.

```lean
-- From DominikPeters/SocialChoiceLean/SocialChoice/Rules/SplitCycle/Defs.lean

/-- Defaite de Split Cycle : x defait y si :
    1. La marge de x sur y est positive (x bat y en duel)
    2. Il n'existe pas de cycle contenant x et y ou toutes les arretes
       ont une marge >= celle de x sur y

    Intuition : on ne retient la defaite que si elle n'est pas contredite
    par un cycle plus fort. -/
noncomputable def splitCycleDefeats {V A : Type} [Fintype V] [Fintype A]
    (P : Profile V A) (x y : A) : Prop :=
  margin_pos P x y ∧
    ¬ ∃ c : List A, x ∈ c ∧ y ∈ c ∧
      cycle (fun a b => margin P x y ≤ margin P a b) c

/-- Split Cycle : les gagnants sont ceux qui ne sont defeats par personne -/
@[scRule]
noncomputable def splitCycle : VotingRule := by
  intro V A _ _ P
  exact Finset.univ.filter (fun x => ∀ y, ¬ splitCycleDefeats P y x)
```

### Axiomes verifiees pour Split Cycle

| Axiome | Statut | Fichier |
|--------|--------|--------|
| Condorcet Consistency | Prouve | `SplitCycle/Condorcet.lean` |
| Monotonicite | Prouve | `SplitCycle/Monotonicity.lean` |
| Pareto | Prouve | `SplitCycle/Pareto.lean` |
| Neutrite | Prouve | `SplitCycle/Neutrality.lean` |
| Smith | Prouve | `SplitCycle/Smith.lean` |
| Clones | Prouve | `SplitCycle/Clones.lean` |
| Independance | Prouve | `SplitCycle/Independence.lean` |
| Renversement | Prouve | `SplitCycle/Reversal.lean` |

---

## 6. Tableau Comparatif des Regles

Les 15+ regles formalisees et leurs proprietes verifiees :

In [1]:
# Tableau comparatif des regles de vote formalisees
# (Python - representation visuelle)

rules = {
    "Split Cycle": {
        "Condorcet": True, "Monotonicity": True, "Pareto": True,
        "Neutrality": True, "Smith": True, "Clones": True,
        "Reversal": True, "Independence": True,
        "source": "Rules/SplitCycle/"
    },
    "Schulze": {
        "Condorcet": True, "Monotonicity": True, "Neutrality": True,
        "Smith": True, "Clones": True, "Reversal": True,
        "Transitivity": True,
        "source": "Rules/Schulze/"
    },
    "Black": {
        "Condorcet": True, "Monotonicity": True, "Neutrality": True,
        "Pareto": True, "Reversal": True, "Smith": True,
        "source": "Rules/Black/"
    },
    "Copeland": {
        "Condorcet": True, "Monotonicity": True, "Neutrality": True,
        "Pareto": True, "Reversal": True, "Smith": True,
        "source": "Rules/Copeland/"
    },
    "Minimax": {
        "Condorcet": True, "Monotonicity": True, "Neutrality": True,
        "Pareto": True, "Reversal": True, "Majority": True,
        "source": "Rules/Minimax/"
    },
    "Borda": {
        "Monotonicity": True, "Neutrality": True, "Pareto": True,
        "Participation": True, "Reinforcement": True,
        "source": "Rules/ScoringRules/Borda/"
    },
    "Plurality": {
        "Monotonicity": True, "Majority": True,
        "source": "Rules/ScoringRules/Plurality/"
    },
    "IRV": {
        "CondorcetLoser": True, "Majority": True,
        "MutualMajority": True, "Clones": True,
        "source": "Rules/ScoringElimination/InstantRunoffVoting/"
    },
    "Baldwin": {
        "Condorcet": True, "CondorcetLoser": True, "Smith": True,
        "source": "Rules/ScoringElimination/Baldwin/"
    },
    "Coombs": {
        "Condorcet": True, "CondorcetLoser": True, "Majority": True,
        "source": "Rules/ScoringElimination/Coombs/"
    },
    "Top Cycle": {
        "Condorcet": True, "Monotonicity": True, "Smith": True,
        "MutualMajority": True,
        "source": "Rules/TopCycle/"
    },
    "Uncovered Set": {
        "Condorcet": True, "Monotonicity": True, "Neutrality": True,
        "Smith": True, "Clones": True,
        "source": "Rules/UncoveredSet/"
    },
}

print(f"Total: {len(rules)} regles formalisees")
print(f"Total: {len(set(ax for r in rules.values() for ax in r if ax != 'source'))} axiomes distincts verifiees")
print()
for name, props in rules.items():
    axioms = sorted(k for k, v in props.items() if k != 'source' and v)
    print(f"  {name:15s} : {len(axioms)} axiomes - {', '.join(axioms)}")

Total: 12 regles formalisees
Total: 14 axiomes distincts verifiees

  Split Cycle     : 8 axiomes - Clones, Condorcet, Independence, Monotonicity, Neutrality, Pareto, Reversal, Smith
  Schulze         : 7 axiomes - Clones, Condorcet, Monotonicity, Neutrality, Reversal, Smith, Transitivity
  Black           : 6 axiomes - Condorcet, Monotonicity, Neutrality, Pareto, Reversal, Smith
  Copeland        : 6 axiomes - Condorcet, Monotonicity, Neutrality, Pareto, Reversal, Smith
  Minimax         : 6 axiomes - Condorcet, Majority, Monotonicity, Neutrality, Pareto, Reversal
  Borda           : 5 axiomes - Monotonicity, Neutrality, Pareto, Participation, Reinforcement
  Plurality       : 2 axiomes - Majority, Monotonicity
  IRV             : 4 axiomes - Clones, CondorcetLoser, Majority, MutualMajority
  Baldwin         : 3 axiomes - Condorcet, CondorcetLoser, Smith
  Coombs          : 3 axiomes - Condorcet, CondorcetLoser, Majority
  Top Cycle       : 4 axiomes - Condorcet, Monotonicity, MutualM

---

## 7. Theoreme de Duggan-Schwartz

Extension de Gibbard-Satterthwaite aux regles multi-gagnants :

```lean
-- From DominikPeters/SocialChoiceLean/SocialChoice/Axioms/Strategyproofness.lean

/-- Manipulabilite optimiste : un electeur peut ameliorer le MEILLEUR
    resultat possible en mentant sur ses preferences. -/
@[scAxiom]
def OptimistStrategyproof (f : VotingRule) : Prop :=
  ∀ {V A} [Fintype V] [Fintype A]
      (P : Profile V A) (v : V) (ballot : LinearOrder A),
    ¬∃ y ∈ f (updateProfile P v ballot),
        ∀ x ∈ f P, Prefers P v y x

/-- Manipulabilite pessimiste : un electeur peut ameliorer le PIRE
    resultat possible en mentant sur ses preferences. -/
@[scAxiom]
def PessimistStrategyproof (f : VotingRule) : Prop :=
  ∀ {V A} [Fintype V] [Fintype A]
      (P : Profile V A) (v : V) (ballot : LinearOrder A),
    ¬∃ x ∈ f P,
        ∀ y ∈ f (updateProfile P v ballot), Prefers P v y x

-- Duggan-Schwartz : si une regle multi-gagnants est non-triviale,
-- surjective, optimist-sp et pessimist-sp, alors elle a un "nominating set"
-- (une coalition de <= 3 electeurs qui controlent le resultat)
-- Formalise dans SocialChoice.Impossibilities.DugganSchwartz.Main
```

---

## 8. Construction et Verification

Le projet Lake `social_choice_lean_peters/` reference DominikPeters comme dependance.

In [2]:
# Verification que le projet Peters existe et a les bons fichiers
import os

peters_dir = "social_choice_lean_peters"
expected_files = ["lakefile.lean", "lean-toolchain", "PetersTour.lean"]

for f in expected_files:
    path = os.path.join(peters_dir, f)
    exists = os.path.exists(path)
    print(f"  {f}: {'OK' if exists else 'MANQUANT'}")

print()
print("Pour construire le projet (dans WSL) :")
print("  cd social_choice_lean_peters")
print("  ~/.elan/bin/lake exe cache get   # Telecharger le cache Mathlib")
print("  ~/.elan/bin/lake build PetersTour  # Construire le tour")

  lakefile.lean: OK
  lean-toolchain: OK
  PetersTour.lean: OK

Pour construire le projet (dans WSL) :
  cd social_choice_lean_peters
  ~/.elan/bin/lake exe cache get   # Telecharger le cache Mathlib
  ~/.elan/bin/lake build PetersTour  # Construire le tour


---

## 9. Synthese

### Contributions de DominikPeters/SocialChoiceLean

1. **Couverture** : La plus grande bibliotheque de theorie du choix social en Lean 4
2. **Praticite** : Les regles de vote sont directement testables (marges calculables)
3. **Meta-framework** : `@[scAxiom]` et `@[scRule]` permettent une verification systematique
4. **Preuves constructives** : Les impossibilites exhibent des contre-exemples

### Integration avec notre travail

| Notre depot | DominikPeters | Integration |
|-------------|---------------|-------------|
| Arrow.lean (Geanakoplos 2005) | Pas de preuve d'Arrow directe | Complementaire |
| Sen.lean (1970) | Pas de preuve de Sen | Complementaire |
| Voting.lean (median voter, Condorcet) | Condorcet axioms, Split Cycle | Phase 3 integration |
| - | Gibbard-Satterthwaite | Reference seule |
| - | 15+ voting rules | Reference seule |

**Phase 3** (future) : Porter les definitions uniques de DominikPeters dans notre
framework `PrefOrder`-base (4 impossibilites Condorcet, scoring rules).

---

## References

- DominikPeters, *SocialChoiceLean*, https://github.com/DominikPeters/SocialChoiceLean (MIT)
- Gibbard, A. (1973). "Manipulation of Voting Schemes"
- Satterthwaite, M. (1975). "Strategy-proofness and Arrow's conditions"
- Holliday, W. & Pacuit, E. (2023). "Split Cycle"
- Schulze, M. (2011). "A new monotonic, clone-independent, reversal symmetric, and Condorcet-consistent single-winner election method"
- Moulin, H. (1988). "Condorcet's principle implies the no show paradox"
- Young, H.P. (1975). "Social choice scoring functions"